# Kaggle run session (resubmission)

One notebook for every training session on both accounts. Edit **only the settings cell**, then
**Save Version -> Save & Run All (Commit)**.

**Kaggle settings:** Accelerator **GPU T4 x2**, Internet **on**.
**Inputs:**
- Tiny-ImageNet: dataset `akash2sharma/tiny-imagenet` (needed by every `tin` run).
- Earlier sessions of the **same account** (Add Input -> *Notebook Output Files* -> this notebook's
  previous versions): needed whenever a KD run's teacher was trained in an earlier session. Teachers are
  found automatically under `/kaggle/input`.

Runs are IDs from `configs/run_plan.yaml` (`python tools/run_queue.py --list` prints them). A run that
cannot finish before `DEADLINE_HOURS` is not started and is reported as *deferred*: put it in the next session.

Outputs: `/kaggle/working/runs/<experiment>/` (kept as notebook output, checkpoints included, so later
sessions can attach it) and `/kaggle/working/<SESSION_NAME>_results.zip` (everything except checkpoints)
to download and place in the repo's `kaggle/` folder.

In [ ]:
# ---- Settings: edit for each session ---------------------------------------------------------
SESSION_NAME = "A_s1"                 # account letter + session number, used for the zip name
QUEUE_GPU0 = ["A1", "A2", "A3", "A4"]  # run IDs in order, on GPU 0
QUEUE_GPU1 = []                        # second queue on GPU 1; leave empty to run one job at a time
DEADLINE_HOURS = 11.0                  # Kaggle stops a session at 12 h
REPO_URL = "https://github.com/nvhungus/Lightweight_Context-Cluster.git"
REPO_BRANCH = "resubmission"

In [ ]:
import json, os, subprocess, sys, time, zipfile
from pathlib import Path

WORK = Path("/kaggle/working")
ROOT = WORK / "Lightweight-Context-Cluster"
RUNS = WORK / "runs"
if not ROOT.exists():
    subprocess.run(["git", "clone", "--depth", "1", "--branch", REPO_BRANCH, REPO_URL, str(ROOT)], check=True)
os.chdir(ROOT)
print("commit:", subprocess.run(["git", "log", "-1", "--format=%H %s"], capture_output=True, text=True).stdout)
print(subprocess.run(["nvidia-smi", "--query-gpu=index,name,memory.total", "--format=csv"], capture_output=True, text=True).stdout)

import torch
sys.path.insert(0, str(ROOT / "tools"))
import run_queue

N_GPUS = torch.cuda.device_count()
assert N_GPUS >= 1 and "T4" in torch.cuda.get_device_name(0), "Select the GPU T4 x2 accelerator."
if QUEUE_GPU1 and N_GPUS < 2:
    raise SystemExit("QUEUE_GPU1 is set but only one GPU is visible: select GPU T4 x2.")

plan = run_queue.load_plan()
ids = QUEUE_GPU0 + QUEUE_GPU1
unknown = [i for i in ids if i not in plan["by_id"]]
assert not unknown, f"unknown run IDs: {unknown}"
assert len(ids) == len(set(ids)), "a run ID appears twice"
for name, queue in [("GPU0", QUEUE_GPU0), ("GPU1", QUEUE_GPU1)]:
    if queue:
        hours = sum(run_queue.estimated_hours(plan, plan["by_id"][i]) or 0 for i in queue)
        print(f"{name}: {queue}  estimated {hours:.1f} h" + ("  (> deadline: the last runs will be deferred)" if hours * 1.3 > DEADLINE_HOURS else ""))


def find_tin_root() -> Path | None:
    for depth in range(1, 7):
        for hit in Path("/kaggle/input").glob("/".join(["*"] * depth) + "/wnids.txt"):
            if (hit.parent / "train").is_dir() and (hit.parent / "val" / "val_annotations.txt").exists():
                return hit.parent
    return None


needs_tin = any(plan["by_id"][i].dataset == "tin" for i in ids)
TIN_ROOT = find_tin_root()
assert TIN_ROOT or not needs_tin, "Attach the Tiny-ImageNet dataset (akash2sharma/tiny-imagenet)."
CIFAR_ROOT = WORK / "data"
print("TIN_ROOT =", TIN_ROOT)

In [ ]:
def queue_cmd(queue, gpu, workers):
    return [sys.executable, "tools/run_queue.py", "--ids", *queue, "--output", str(RUNS),
            "--tin-root", str(TIN_ROOT), "--cifar-root", str(CIFAR_ROOT), "--search-dir", "/kaggle/input",
            "--gpu", str(gpu), "--workers", str(workers), "--deadline-hours", str(DEADLINE_HOURS), "--keep-going"]


# One queue: 4 data workers (as in the first-submission timings). Two queues share 4 CPUs: 2 each.
workers = 2 if QUEUE_GPU1 else 4
jobs = []
for gpu, queue in [(0, QUEUE_GPU0), (1, QUEUE_GPU1)]:
    if queue:
        log = open(WORK / f"{SESSION_NAME}_gpu{gpu}.log", "w")
        jobs.append((gpu, subprocess.Popen(queue_cmd(queue, gpu, workers), cwd=ROOT, stdout=log, stderr=subprocess.STDOUT, text=True), log))
        print(f"started GPU{gpu}: {queue}")

start = time.time()
while any(p.poll() is None for _, p, _ in jobs):
    time.sleep(600)
    for gpu, p, _ in jobs:
        lines = (WORK / f"{SESSION_NAME}_gpu{gpu}.log").read_text(errors="replace").splitlines()
        progress = [l for l in lines if l.startswith(("===", "epoch=", "test ", "[defer]", "[skip]", "[fail]"))]
        print(f"[{(time.time() - start) / 3600:5.2f} h] GPU{gpu}: {progress[-1] if progress else '(starting)'}", flush=True)
for gpu, p, log in jobs:
    log.close()
    print(f"\n===== GPU{gpu} queue finished with exit code {p.returncode} =====")
    text = (WORK / f"{SESSION_NAME}_gpu{gpu}.log").read_text(errors="replace")
    print(text[text.rfind("=== queue summary ==="):] if "=== queue summary ===" in text else text[-3000:])

In [ ]:
# ---- Session summary -------------------------------------------------------------------------
rows = []
for run_id in ids:
    run = plan["by_id"][run_id]
    d = RUNS / run.experiment
    test, info = d / "test_metrics.json", d / "run_info.json"
    if test.exists():
        t = json.loads(test.read_text())
        i = json.loads(info.read_text())
        rows.append((run_id, run.experiment, "done", i.get("best_val_acc1"), t.get("test_acc1"), i.get("best_epoch"), i.get("git_commit", "")[:7]))
    else:
        rows.append((run_id, run.experiment, "NOT DONE", None, None, None, ""))
print(f"{'id':>4} {'experiment':38s} {'status':9s} {'best val':>9s} {'test':>7s} {'best ep':>8s} commit")
for r in rows:
    fmt = lambda v: f"{v:.2f}" if isinstance(v, float) else "-"
    print(f"{r[0]:>4} {r[1]:38s} {r[2]:9s} {fmt(r[3]):>9s} {fmt(r[4]):>7s} {str(r[5]) if r[5] is not None else '-':>8s} {r[6]}")
not_done = [r[0] for r in rows if r[2] != "done"]
print("\nAll runs finished." if not not_done else f"\nNot finished (re-queue in the next session): {not_done}")

bundle = WORK / f"{SESSION_NAME}_results.zip"
with zipfile.ZipFile(bundle, "w", zipfile.ZIP_DEFLATED) as zf:
    for f in RUNS.rglob("*"):
        if f.is_file() and f.suffix != ".pth":
            zf.write(f, f.relative_to(WORK))
    for f in WORK.glob(f"{SESSION_NAME}_gpu*.log"):
        zf.write(f, f.relative_to(WORK))
print("download:", bundle)